In [3]:
import json
import psycopg2
from kafka import KafkaConsumer
from sentence_transformers import SentenceTransformer


KAFKA_TOPIC = 'transactions'
KAFKA_BOOTSTRAP_SERVERS = 'kafka_streaming_lab:9092'
DB_CONFIG = {
    "host": "host.docker.internal",
    "database": "nosql_lab_database",
    "user": "postgres",
    "password": "postgres",
    "port": 5433
}


model = SentenceTransformer('all-MiniLM-L6-v2') 

conn = psycopg2.connect(**DB_CONFIG)
cursor = conn.cursor()


try:
    print("Sprawdzam strukturę bazy...")
    cursor.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    # Tworzymy tabelę
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS transactions (
            id SERIAL PRIMARY KEY,
            sender VARCHAR(50),
            receiver VARCHAR(50),
            amount NUMERIC(10, 2),
            timestamp TIMESTAMP,
            device_sender VARCHAR(20),
            device_receiver VARCHAR(20),
            title TEXT,
            title_embedding vector(384)
        );
    """)
    conn.commit()
    print("Baza przygotowana.")
except Exception as e:
    print(f"Błąd przy przygotowaniu bazy: {e}")
    conn.rollback()

# Konsument Kafki
consumer = KafkaConsumer(
    KAFKA_TOPIC,
    bootstrap_servers=[KAFKA_BOOTSTRAP_SERVERS],
    value_deserializer=lambda v: json.loads(v.decode('utf-8')),
    auto_offset_reset='earliest'
)

print("Konsument uruchomiony, czekam na transakcje...")

try:
    for message in consumer:
        tx = message.value
        embedding = model.encode(tx['title']).tolist()
        
        insert_query = """
            INSERT INTO transactions (sender, receiver, amount, timestamp, device_sender, device_receiver, title, title_embedding)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """
        
        cursor.execute(insert_query, (
            tx['sender'], 
            tx['receiver'], 
            tx['amount'], 
            tx['timestamp'], 
            tx['device_sender'], 
            tx['device_receiver'], 
            tx['title'], 
            embedding
        ))
        
        conn.commit()
        print(f"Zapisano transakcję: {tx['title']}")

except Exception as e:
    print(f"Błąd podczas pracy konsumenta: {e}")
    conn.rollback()
finally:
    cursor.close()
    conn.close()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Sprawdzam strukturę bazy...
Błąd przy przygotowaniu bazy: BŁĄD:  extension "vector" is not available
HINT:  The extension must first be installed on the system where PostgreSQL is running.

Konsument uruchomiony, czekam na transakcje...
Błąd podczas pracy konsumenta: BŁĄD:  relacja "transactions" nie istnieje
LINE 2:             INSERT INTO transactions (sender, receiver, amou...
                                ^

